In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

###This is the raw incentive data which is used here to transform and clear out the issues fetch the json data out of it

raw incentive data

In [0]:
raw_incentive_data=spark.read.table('johndeere.default.raw_incentives_sales_bronze')
display(raw_incentive_data)

In [0]:
raw_incentive_data.printSchema()

## dropping rescued data for now

In [0]:
raw_incentive_1=raw_incentive_data.drop('_res_data','_rescued_data')

In [0]:
raw_incentive_1.printSchema()

## extracting data from json fields and creating a final table out of it


In [0]:
clean_raw_incentives_1=raw_incentive_1.withColumn('progs_array',explode_outer(col('progs')))\
    
# display(clean_raw_incentives_1)
# clean_raw_incentives_1.printSchema()
clean_raw_incentive_2=clean_raw_incentives_1.select(col('dealer_id'),
                                                    col('sale_id'),
                                                    col('tx_date'),
                                                    col('progs_array.p_name').alias("p_name"),
                                                    col('progs_array.perc').alias("percentage"),
                                                    col('equip.m_id').alias("m_id"),
                                                    col('equip.price').alias("price"))
                                                    
display(clean_raw_incentive_2)

## dropping duplicates

In [0]:
clean_raw_incentive_2=clean_raw_incentive_2.dropDuplicates()



## quarantine raw incentive dataset (dataset which has null dealer id)

In [0]:


quarantine_raw_incentives = clean_raw_incentive_2.filter(
    col("dealer_id").isNull() |
    (trim(col("dealer_id")) == "")
)

display(quarantine_raw_incentives)



## quarantine table / dead tables records

In [0]:
quarantine_raw_incentives.write\
  .mode('append')\
    .saveAsTable('johndeere.quarantine.raw_incentives_quarantined')

## Final cleansed Version of raw incentive dataset

In [0]:
raw_incentives_final = clean_raw_incentive_2.filter(
    col("dealer_id").isNotNull() &
    (trim(col("dealer_id")) != "")
)

display(raw_incentives_final)

In [0]:
raw_incentives_final=raw_incentives_final.replace('UNKNOWN','2026-03-15',subset=['tx_date'])

In [0]:

raw_incentives_final = raw_incentives_final.withColumn(
    "tx_date",
    coalesce(
        expr("try_to_date(tx_date, 'yyyy-MM-dd')"),
        expr("try_to_date(tx_date, 'yyyy/MM/dd')"),
        expr("try_to_date(tx_date, 'yyyy.MM.dd')"),
        expr("try_to_date(tx_date, 'dd-MM-yyyy')")
    )
)


## passing the dataframe on to table

In [0]:

raw_incentives_final.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("johndeere.cleansed_tables.raw_incentives_dataset")


## dealers dimension dataframe

In [0]:
dim_dealers_bronze=spark.read.table('johndeere.default.dim_dealers_bronze_table')
display(dim_dealers_bronze)

## dropping fields not required 

In [0]:
dim_dealers_2=dim_dealers_bronze.drop('equip','progs','sale_id','tx_date','_res_data')
#display(dim_dealers_2)

## saving quarantined dealer dimension table into db

In [0]:
quarantine_dim_dealers=dim_dealers_2.filter(col('tier').isNull())
#display(quarantine_dim_dealers)
quarantine_dim_dealers.write\
    .mode('append')\
        .saveAsTable('johndeere.quarantine.dim_dealers_quarantined')

In [0]:
dim_dealers_cleaned=dim_dealers_2.filter(col('tier').isNotNull())
#display(dim_dealers_cleaned)
dim_dealers_cleaned.write\
    .mode('append')\
        .saveAsTable('johndeere.cleansed_tables.dim_dealers_dataset')

## dim equipments bronze

In [0]:
dim_equipments_bronze=spark.read.table('johndeere.default.dim_equipments_bronze_table')
dim_equipments_bronze=dim_equipments_bronze.drop('_res_data')
# display(dim_equipments_bronze)
dim_equipments_bronze_2=dim_equipments_bronze.dropDuplicates()
# dim_equipments_bronze_2.groupBy("model_id").agg(count('*')).sort("model_id").display()
display(dim_equipments_bronze_3)
dim_equipments_bronze_3=dim_equipments_bronze_2.withColumn('category',upper(col("category")))


## quarantined dim equipments and storing it into tables

In [0]:
quarantined_dim_equipments=dim_equipments_bronze_3.filter(col('base_price')<=0)
quarantined_dim_equipments.write\
  .mode('append')\
    .saveAsTable('johndeere.quarantine.dim_equipments_quarantined')

## equipments cleansed

In [0]:
dim_equipments_final=dim_equipments_bronze_3.filter(col('base_price')>0)
dim_equipments_final.write\
    .mode('append')\
        .saveAsTable('johndeere.cleansed_tables.dim_equipments_dataset')

## legacy_payout dataframe

In [0]:
legacy_bronze=spark.read.table('johndeere.default.legacy_payout_bronze_table')
display(legacy_bronze)


In [0]:
legacy_bronze=legacy_bronze.drop('_res_data')
display(legacy_bronze)

In [0]:
legacy_bronze_2=legacy_bronze.dropDuplicates()
# display(legacy_bronze_2)
# legacy_bronze_2.filter(lower(col('payout_status'))=='success').display()
legacy_bronze_2.write\
    .mode('append')\
        .saveAsTable('johndeere.cleansed_tables.legacy_dataset')

##Schemas 



In [0]:
legacy_bronze_2.printSchema()

In [0]:
raw_incentives_final.printSchema()

In [0]:
dim_dealers_cleaned.printSchema()

In [0]:
dim_equipments_final.printSchema()